In [ ]:
pip install bayesian-optimization

In [ ]:
import numpy as np
from bayes_opt import BayesianOptimization

n = 5
mu = np.array([
    8.6033358901938017e-01, 3.4256184594817283e+00, 6.4372981791719468e+00,
    9.5293344053619631e+00, 1.2645287223856643e+01, 1.5771284874815882e+01,
    1.8902409956860023e+01, 2.2036496727938566e+01, 2.5172446326646664e+01,
    2.8309642854452012e+01, 3.1447714637546234e+01, 3.4586424215288922e+01,
    3.7725612827776501e+01, 4.0865170330488070e+01, 4.4005017920830845e+01,
    4.7145097736761031e+01, 5.0285366337773652e+01, 5.3425790477394663e+01,
    5.6566344279821521e+01, 5.9707007305335459e+01, 6.2847763194454451e+01,
    6.5988598698490392e+01, 6.9129502973895256e+01, 7.2270467060308960e+01,
    7.5411483488848148e+01, 7.8552545984242926e+01, 8.1693649235601683e+01,
    8.4834788718042290e+01, 8.7975960552493220e+01, 9.1117161394464745e+01
])
x = np.ones(5)  /np.sqrt(5)
def rho(x):
    """Calculate the rho function for given x and mu values."""
    n = len(x)
    result = np.zeros(30)
    for j in range(30):
        exp_term = np.exp(-mu[j]**2 * np.sum(x**2))
        sum_term = sum(2 * (-1)**(ii-1) * np.exp(-mu[j]**2 * np.sum(x[ii-1:]**2)) for ii in range(2, n+1))
        result[j] = -(exp_term + sum_term + (-1)**n) / mu[j]**2
    return result

def A(mu_val):
    """Calculate the A function for a given mu value."""
    return 2 * np.sin(mu_val) / (mu_val + np.sin(mu_val) * np.cos(mu_val))

A_values= A(mu)

#BAYESIAN
def black_box(mu0, mu1, x):

    A_values = A(mu)
    mu_modified = mu.copy()
    mu_modified[0] = mu0
    mu_modified[1] = mu1

    rho_values = rho(x)

    term1 = sum(mu_modified[i]**2 * mu_modified[j]**2 * A_values[i] * A_values[j] * rho_values[i] * rho_values[j] *
                (np.sin(mu_modified[i]+mu_modified[j])/(mu_modified[i]+mu_modified[j]) + np.sin(mu_modified[i]-mu_modified[j])/(mu_modified[i]-mu_modified[j]))
                for i in range(30) for j in range(i+1, 30))

    term2 = sum(mu_modified[j]**4 * A_values[j]**2 * rho_values[j]**2 *
                (np.sin(2*mu_modified[j])/(2*mu_modified[j]) + 1)/2 for j in range(30))

    term3 = sum(mu_modified[j]**2 * A_values[j] * rho_values[j] *
                (2*np.sin(mu_modified[j])/mu_modified[j]**3 - 2*np.cos(mu_modified[j])/mu_modified[j]**2)
                for j in range(30))

    return (term1 + term2 - term3 + 2/15 - 0.0001)

pbounds = {
    'mu0': (mu[0] * -0.5, mu[0] * 1.2),
    'mu1': (mu[1] * -0.5, mu[1] * 1.2)
}  # Optimize only selected indices

def bo_wrapper(mu0, mu1):
    return black_box(mu0, mu1, x)

optimizer = BayesianOptimization(f=bo_wrapper, pbounds=pbounds, random_state=42)

# Run Bayesian Optimization
optimizer.maximize(init_points=5, n_iter=10)

# Print best result
print(optimizer.max["target"])

|   iter    |  target   |    mu0    |    mu1    |
-------------------------------------------------
| 1         | 0.1619    | 0.1176    | 3.824     |
| 2         | 0.1128    | 0.6404    | 1.774     |
| 3         | 0.1375    | -0.202    | -0.8044   |
| 4         | 0.1526    | -0.3452   | 3.331     |
| 5         | 0.1397    | 0.449     | 2.411     |
| 6         | 0.1245    | 0.657     | 3.448     |
| 7         | 0.08288   | 0.7815    | 0.03587   |
| 8         | 0.1508    | -0.3476   | 3.969     |
| 9         | 0.1252    | -0.4219   | 0.988     |
| 10        | 0.1431    | -0.4234   | 2.545     |
| 11        | 0.1346    | -0.4152   | -1.692    |
| 12        | 0.1456    | 0.4096    | 4.11      |
| 13        | 0.06637   | 1.015     | -1.692    |
| 14        | 0.1188    | -0.428    | -0.01869  |
| 15        | 0.1644    | 0.05789   | 3.015     |
0.16435247664896602


In [ ]:
import numpy as np
from scipy.optimize import minimize
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
import csv
from scipy import linalg as LA

#FGN : The problem expressed here in form   min f(x) : c(x)>=0

# Define the problem parameters
n = 5
mu = np.array([
    8.6033358901938017e-01, 3.4256184594817283e+00, 6.4372981791719468e+00,
    9.5293344053619631e+00, 1.2645287223856643e+01, 1.5771284874815882e+01,
    1.8902409956860023e+01, 2.2036496727938566e+01, 2.5172446326646664e+01,
    2.8309642854452012e+01, 3.1447714637546234e+01, 3.4586424215288922e+01,
    3.7725612827776501e+01, 4.0865170330488070e+01, 4.4005017920830845e+01,
    4.7145097736761031e+01, 5.0285366337773652e+01, 5.3425790477394663e+01,
    5.6566344279821521e+01, 5.9707007305335459e+01, 6.2847763194454451e+01,
    6.5988598698490392e+01, 6.9129502973895256e+01, 7.2270467060308960e+01,
    7.5411483488848148e+01, 7.8552545984242926e+01, 8.1693649235601683e+01,
    8.4834788718042290e+01, 8.7975960552493220e+01, 9.1117161394464745e+01
])

def objective(x):
    """Calculate the objective function value."""
    return np.sum(x**2)

def rho(x):
    """Calculate the rho function for given x and mu values."""
    n = len(x)
    result = np.zeros(30)
    for j in range(30):
        exp_term = np.exp(-mu[j]**2 * np.sum(x**2))
        sum_term = sum(2 * (-1)**(ii-1) * np.exp(-mu[j]**2 * np.sum(x[ii-1:]**2)) for ii in range(2, n+1))
        result[j] = -(exp_term + sum_term + (-1)**n) / mu[j]**2
    return result

def A(mu_val):
    """Calculate the A function for a given mu value."""
    return 2 * np.sin(mu_val) / (mu_val + np.sin(mu_val) * np.cos(mu_val))

#BAYESIAN
def black_box(mu0, mu1, x):
    A_values = A(mu)
    mu_modified = mu.copy()
    mu_modified[0] = mu0
    mu_modified[1] = mu1

    rho_values = rho(x)

    term1 = sum(mu_modified[i]**2 * mu_modified[j]**2 * A_values[i] * A_values[j] * rho_values[i] * rho_values[j] *
                (np.sin(mu_modified[i]+mu_modified[j])/(mu_modified[i]+mu_modified[j]) + np.sin(mu_modified[i]-mu_modified[j])/(mu_modified[i]-mu_modified[j]))
                for i in range(30) for j in range(i+1, 30))

    term2 = sum(mu_modified[j]**4 * A_values[j]**2 * rho_values[j]**2 *
                (np.sin(2*mu_modified[j])/(2*mu_modified[j]) + 1)/2 for j in range(30))

    term3 = sum(mu_modified[j]**2 * A_values[j] * rho_values[j] *
                (2*np.sin(mu_modified[j])/mu_modified[j]**3 - 2*np.cos(mu_modified[j])/mu_modified[j]**2)
                for j in range(30))

    return (term1 + term2 - term3 + 2/15 - 0.0001)

def constraint(x, mu, A_values):
    """Calculate the constraint function value."""
    # Bayesian Optimization
    pbounds = {
        'mu0': (mu[0] * -0.5, mu[0] * 1.2),
        'mu1': (mu[1] * -0.5, mu[1] * 1.2)
    }  # Optimize only selected indices

    def bo_wrapper(mu0, mu1):
        return black_box(mu0, mu1, x)

    optimizer = BayesianOptimization(f=bo_wrapper, pbounds=pbounds, verbose = 0)

    # Run Bayesian Optimization
    optimizer.maximize(init_points=5, n_iter=10)
    print("Max constr değeri : ", optimizer.max["target"])
    # Print best result
    return -1 * optimizer.max["target"]

def generate_data(x, radius, mu, A_values, n_samples):
    """Generate data points around current point x within given radius."""
    X = []
    y_obj = []
    y_const = []
    for _ in range(n_samples):
        delta = np.random.uniform(-radius, radius, n)
        x_new = x + delta
        X.append(x_new)
        y_obj.append(objective(x_new))
        y_const.append(constraint(x_new, mu, A_values))
    return np.array(X), np.array(y_obj), np.array(y_const)



def trust_region_optimization(x0, mu, A_values, max_iterations=69, initial_radius=1.0, initial_sample = 10, new_generation= 1):
    """Main trust region optimization function with weight decay calibration."""

    x = x0
    radius = initial_radius
    results = []
    penalty_factor = 1000  # Penalty for constraint violations
    X_cumulative = []
    y_obj_cumulative = []
    y_const_cumulative = []

    # Lists to store values for output
    constraint_values = []
    predicted_constraint_values = []
    objective_values = []
    predicted_objective_values = []

    # Generate initial dataset
    X_init, y_obj_init, y_const_init = generate_data(x, radius, mu, A_values, n_samples = initial_sample)
    X_cumulative.extend(X_init)
    y_obj_cumulative.extend(y_obj_init)
    y_const_cumulative.extend(y_const_init)
    count = 0

    #FGN : Evaluate functions at the initial point and store
    current_obj = objective(x0)
    current_cons = constraint(x0, mu, A_values)
    X_cumulative.append(x0)
    y_obj_cumulative.append(current_obj)
    y_const_cumulative.append(current_cons)

    for iteration in range(max_iterations):
        # Generate one new data point
        for _ in range(new_generation):
          delta = np.random.uniform(-radius, radius, n)
          x_new = x + delta

          X_cumulative.append(x_new)
          y_obj_cumulative.append(objective(x_new))
          y_const_cumulative.append(constraint(x_new, mu, A_values))

        # Calculate adaptive weights with calibrated decay
        distances = np.linalg.norm(np.array(X_cumulative) - x, axis=1)
        alpha = 0.25  # Decay rate parameter, adjust based on problem
        weights = np.where(distances >= 6*radius, 0, np.exp(-alpha * (distances)**2))
        #weights = weights*(1/np.linalg.norm(weights,ord=np.inf))
        #weights = np.ones(np.shape(weights))
        # Create a dictionary pairing distances with weights
        distance_weight_dict = {f"Data point {i+1}": {"Distance": distances[i], "Weight": weights[i]}
                            for i in range(len(distances))}

        # Log distances and weights
        print(f"Iteration {iteration}:")
        #for key, value in distance_weight_dict.items():
        #  print(f"{key}: Distance = {value['Distance']:.6f}, Weight = {value['Weight']:.6f}")

        # Transform the data using PolynomialFeatures
        poly = PolynomialFeatures(degree=2, include_bias=True)
        X_transformed = poly.fit_transform(np.array(X_cumulative))

        # Fit objective and constraint models
        objective_model = LinearRegression()
        constraint_model = LinearRegression()

        objective_model.fit(X_transformed, np.array(y_obj_cumulative), sample_weight=weights)
        constraint_model.fit(X_transformed, np.array(y_const_cumulative), sample_weight=weights)

        # Define surrogate subproblem
        def subproblem_objective(delta):
            delta_transformed = poly.transform([x + delta])
            return objective_model.predict(delta_transformed)[0]

        def subproblem_constraint(delta):
            delta_transformed = poly.transform([x + delta])
            return constraint_model.predict(delta_transformed)[0]

        # Solve subproblem
        res = minimize(subproblem_objective, np.zeros_like(x), method='SLSQP',
                      constraints={'type': 'ineq', 'fun': subproblem_constraint},
                      bounds=[(-radius, radius)] * len(x))
        # FGN: Don't you check if minimization is successful (the return code)?
        delta = res.x

        # Calculate actual-to-predicted ratio (p_k)
        real_constraint_value = constraint(x + delta, mu, A_values)
        predicted_constraint_value = subproblem_constraint(delta)
        real_objective_value = objective(x + delta)
        predicted_objective_value = subproblem_objective(delta)

        px = current_obj + penalty_factor * max(0, -1 * current_cons)
        pxx = objective(x + delta) + penalty_factor * max(0, -1 * constraint(x + delta, mu, A_values))

        mp = subproblem_objective(0) + penalty_factor * max(0, -1 * subproblem_constraint(0))
        mpp = subproblem_objective(delta) + penalty_factor * max(0, -1 * subproblem_constraint(delta))

        pk = (px - pxx) / (mp - mpp) if (mp - mpp) != 0 else 0
        print("px=",px)
        print("mp=",mp)
        print("pxx=",pxx)
        print("mpp=",mpp)
        # Adjust trust region radius
        #if pk <= 0.1:
        #    count += 1
        #if count >= 10:
        #    radius = initial_radius
        #    count = 0
        if pk >= 0.6:
            radius = min(2 * radius, 2.0)
        elif pk < 0.1:
            radius = max(0.5 * radius, 0.001)

        # FGN: Add the new evaluation to the data set
        X_cumulative.append(x+delta)
        y_obj_cumulative.append(real_objective_value)
        y_const_cumulative.append(real_constraint_value)

        # Update solution if improvement is sufficient
        if pk >= 0.1:
            x = x + delta
            #FGN : also store the evaluation values here so that you do not re-evalute at x
            current_obj = real_objective_value
            current_cons = real_constraint_value

        # Log progress
        if iteration % 1 == 0:
            print(f"Iteration {iteration}:")
            print(f"Radius: {radius:.6f}")
            print(f"pk: {pk:.6f}")
            print(f"Objective: {objective(x):.6f}")
            print(f"Real Constraint Value: {-1*real_constraint_value:.6f}")
            print(f"Predicted Constraint Value: {-1*predicted_constraint_value:.6f}")
            print(f"Historical data points: {len(X_cumulative)}")
            print("------------------------")

        # Store values for logging
        constraint_values.append(real_constraint_value)
        predicted_constraint_values.append(predicted_constraint_value)
        objective_values.append(real_objective_value)
        predicted_objective_values.append(predicted_objective_value)

        # Store results
        results.append((iteration, objective(x), x, real_constraint_value, predicted_constraint_value,
                       real_objective_value, predicted_objective_value))

    return results, constraint_values, predicted_constraint_values, objective_values, predicted_objective_values




def save_results(results, filename='optimization_results_polynomial.csv'):
    """Save optimization results to CSV file."""
    with open(filename, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['Iteration', 'Objective Value'] +
                       [f'x{i+1}' for i in range(n)] +
                       ['Real Constraint Value', 'Predicted Constraint Value',
                        'Real Objective Value', 'Predicted Objective Value'])
        for result in results:
            writer.writerow([result[0], result[1]] +
                          list(result[2]) +
                          [result[3], result[4], result[5], result[6]])

def main():
    """Main function to run the optimization."""
    # Calculate A_values once
    A_values = np.array([A(m) for m in mu])

    # Set random seed for reproducibility
    np.random.seed(1234)

    # Initialize starting point
    x0 = np.random.randn(n)
    print("Initial point:", x0)

    # Run optimization
    results, constraint_values, predicted_constraint_values, objective_values, predicted_objective_values = \
        trust_region_optimization(x0, mu, A_values)

    # Save results
    #save_results(results)
    print("Optimization complete. Results saved to 'optimization_results_polynomial.csv'.")

if __name__ == "__main__":
    main()

Initial point: [ 0.47143516 -1.19097569  1.43270697 -0.3126519  -0.72058873]
Max constr değeri :  0.722690406561195
Max constr değeri :  0.21356916418838814
Max constr değeri :  0.657511614579398
Max constr değeri :  0.16115218791594033
Max constr değeri :  0.140436789673647
Max constr değeri :  0.1622579787848937
Max constr değeri :  0.7837765266334085
Max constr değeri :  0.6063375997460752
Max constr değeri :  0.6557469014627977
Max constr değeri :  0.481291212545793
Max constr değeri :  0.17213613324241545
Max constr değeri :  0.9375640497997944
Iteration 0:
Max constr değeri :  0.14312430504887041
Max constr değeri :  0.1380171902288667
px= 176.44645604955952
mp= 176.44645604955915
pxx= 139.5370455956173
mpp= 0.5772298270352344
Iteration 0:
Radius: 1.000000
pk: 0.209868
Objective: 1.519855
Real Constraint Value: 0.143124
Predicted Constraint Value: 0.000000
Historical data points: 13
------------------------
Max constr değeri :  0.17085332823148472
Iteration 1:
Max constr değeri :

/usr/local/lib/python3.11/dist-packages/scipy/optimize/_slsqp_py.py:434: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
/usr/local/lib/python3.11/dist-packages/scipy/optimize/_slsqp_py.py:434: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)


Max constr değeri :  0.11300503721349292
Max constr değeri :  0.13049156857681585
px= 123.5535700139129
mp= 123.5535700136221
pxx= 132.09307387666715
mpp= 109.51736092809527
Iteration 22:
Radius: 0.001000
pk: -0.608391
Objective: 1.601761
Real Constraint Value: 0.113005
Predicted Constraint Value: 0.107916
Historical data points: 57
------------------------
Max constr değeri :  0.13256706121780337
Iteration 23:
Max constr değeri :  0.1383268819412357
Max constr değeri :  0.1249357174111591
px= 123.5535700139129
mp= 123.55357001384948
pxx= 126.53540414308402
mpp= 96.0894964226922
Iteration 23:
Radius: 0.001000
pk: -0.108572
Objective: 1.601761
Real Constraint Value: 0.138327
Predicted Constraint Value: 0.094490
Historical data points: 59
------------------------
Max constr değeri :  0.13041582737840213
Iteration 24:


/usr/local/lib/python3.11/dist-packages/scipy/optimize/_slsqp_py.py:434: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
/usr/local/lib/python3.11/dist-packages/scipy/optimize/_slsqp_py.py:434: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
/usr/local/lib/python3.11/dist-packages/scipy/optimize/_slsqp_py.py:434: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
/usr/local/lib/python3.11/dist-packages/scipy/optimize/_slsqp_py.py:434: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
/usr/local/lib/python3.11/dist-packages/scipy/optimize/_slsqp_py.py:434: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
/usr/local/lib/python3.11/dist-packages/scipy/optimize/_slsqp_py.py:43

Max constr değeri :  0.13787000790361273
Max constr değeri :  0.12021582928629963
px= 123.5535700139129
mp= 123.55357001384948
pxx= 121.81684058314976
mpp= 127.4020476404043
Iteration 24:
Radius: 0.001000
pk: -0.451277
Objective: 1.601761
Real Constraint Value: 0.137870
Predicted Constraint Value: 0.125801
Historical data points: 61
------------------------
Max constr değeri :  0.12915424490875513
Iteration 25:
Max constr değeri :  0.13770971505786864
Max constr değeri :  0.12159353843943191
px= 123.5535700139129
mp= 123.55357001498635
pxx= 123.19800491087369
mpp= 54.85644506226949
Iteration 25:
Radius: 0.001000
pk: 0.005176
Objective: 1.601761
Real Constraint Value: 0.137710
Predicted Constraint Value: 0.053252
Historical data points: 63
------------------------
Max constr değeri :  0.13169521418777153
Iteration 26:


/usr/local/lib/python3.11/dist-packages/scipy/optimize/_slsqp_py.py:434: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)


Max constr değeri :  0.1253957495960453
Max constr değeri :  0.1259066624761777
px= 123.5535700139129
mp= 123.55357002385392
pxx= 127.50634549585641
mpp= 47.18850216994109
Iteration 26:
Radius: 0.001000
pk: -0.051762
Objective: 1.601761
Real Constraint Value: 0.125396
Predicted Constraint Value: 0.045589
Historical data points: 65
------------------------
Max constr değeri :  0.1437102728699492
Iteration 27:
Max constr değeri :  0.12919430784665878
Max constr değeri :  0.1431377360884624
px= 123.5535700139129
mp= 123.553570011121
pxx= 144.7350751994663
mpp= 1.5973391612414571
Iteration 27:
Radius: 0.001000
pk: -0.173681
Objective: 1.601761
Real Constraint Value: 0.129194
Predicted Constraint Value: -0.000001
Historical data points: 67
------------------------
Max constr değeri :  0.1281034635921631
Iteration 28:


KeyboardInterrupt: 